# Customer Silver Data Quality Profiling

The objective of this stage is to identify records that violate
the defined Silver-layer data-quality rules.

No records will be modified or removed during profiling.

Configuration / Paths

In [74]:
from google.colab import drive
drive.mount('/content/drive')

BRONZE_PATH = "/content/drive/MyDrive/pyspark-ecommerce-analytics-data/bronze"
CUSTOMER_PATH = f"{BRONZE_PATH}/customers"
PRODUCT_PATH = f"{BRONZE_PATH}/products"
ORDER_PATH = f"{BRONZE_PATH}/orders"
TRANSACTION_PATH = f"{BRONZE_PATH}/transactions"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Spark Session

In [75]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('silver-data-profiling').getOrCreate()
spark.version

'4.0.4'

Load Data from Bronze


In [76]:
bronze_customers = spark.read.parquet(CUSTOMER_PATH)
bronze_orders = spark.read.parquet(ORDER_PATH)
bronze_products = spark.read.parquet(PRODUCT_PATH)
bronze_transactions = spark.read.parquet(TRANSACTION_PATH)


## Row Counts

In [77]:
print(f"Row Count for Customer: {bronze_customers.count()}")
print(f"Row Count for Products: {bronze_products.count()}")
print(f"Row Count for Orders: {bronze_orders.count()}")
print(f"Row Count for Transaction: {bronze_transactions.count()}")

Row Count for Customer: 10000
Row Count for Products: 1000
Row Count for Orders: 30000
Row Count for Transaction: 50000


## Schema

In [78]:
print('Schema of Customer:')
bronze_customers.printSchema()
print('Schema of Products:')
bronze_products.printSchema()
print('Schema of Orders:')
bronze_orders.printSchema()
print('Schema of Transaction:')
bronze_transactions.printSchema()

Schema of Customer:
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

Schema of Products:
root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



## **Customer Profiling**

In [79]:
bronze_customers.show(5, truncate=False)

+-----------+----------+---------+-------------------------+-------------+-------+-----------+----------------+--------------+--------------------------+--------------------------------------------------------------------------------+
|customer_id|first_name|last_name|email                    |country      |city   |signup_date|customer_segment|account_status|_ingestion_timestamp      |_source_file                                                                    |
+-----------+----------+---------+-------------------------+-------------+-------+-----------+----------------+--------------+--------------------------+--------------------------------------------------------------------------------+
|C000001    |Mia       |Davis    |mia.davis1@example.com   |Germany      |Munich |2024-10-14 |Small Business  |Active        |2026-10-02 14:04:46.631308|file:///content/drive/MyDrive/pyspark-ecommerce-analytics-data/raw/customers.csv|
|C000002    |Arjun     |Davis    |arjun.davis2@example.com |

customer_id

In [80]:
bronze_customers.filter((F.col('customer_id').isNull()) |
 (F.trim(F.col('customer_id')) == '') |
  (F.col('customer_id') != F.trim(F.col('customer_id'))) |
   (F.lower(F.trim(F.col('customer_id'))).isin('unknown', 'n/a', 'na', 'null', 'none'))
   ).count()

0

duplicate customer_id

In [81]:
bronze_customers.groupBy('customer_id').count().filter("count > 1").select('customer_id').count()

1

In [82]:
bronze_customers.filter((F.col('customer_id') != F.upper(F.col('customer_id'))) | ((F.col('customer_id')) != F.trim(F.col('customer_id')))).count()

0

first_name

In [83]:
bronze_customers.filter(
    "(first_name IS NULL) OR (TRIM(first_name) = '') OR (LOWER(TRIM(first_name)) IN ('unknown', 'n/a', 'none', 'null', 'na', '.'))"
).count()

1

last_name


In [84]:
bronze_customers.filter(
    "(last_name IS NULL) OR (TRIM(last_name) = '') OR (LOWER(TRIM(last_name)) IN ('unknown', 'n/a', 'none', 'null', 'na', '.'))"
).count()

0

email

In [85]:
bronze_customers.filter(
    (F.col('email').isNull()) | (~F.col('email').contains('@')) | (~F.col('email').contains('.'))
).count()

1

country

In [86]:
bronze_customers.filter(
    """
    (country IS NULL)
    OR (LOWER(TRIM(country)) = 'unknown')
    OR (TRIM(country) = '')
    """
).count()

1

customer_segment

In [145]:
bronze_customers.select('account_status').distinct().show()

+--------------+
|account_status|
+--------------+
|     Suspended|
|        Active|
|       Dormant|
|      Inactive|
+--------------+



In [144]:
from pyspark.sql import functions as F

invalid_segment_count = bronze_customers.filter(
    (F.col("customer_segment").isNull()) |
    (F.trim(F.col("customer_segment")) == "") |
    (F.lower(F.trim(F.col("customer_segment"))).isin(['unknown', 'n/a', 'none', 'null', 'na', '.'])) |
    (~F.upper(F.trim(F.col("customer_segment"))).isin(['CONSUMER', 'PREMIER', 'ENTERPRISE', 'SMALL BUSINESS']))
).count()

print(f"Invalid customer_segment count: {invalid_segment_count}")

Invalid customer_segment count: 0


account_status

In [148]:
invalid_status_count = bronze_customers.filter(
    (F.col("account_status").isNull()) |
    (F.trim(F.col("account_status")) == "") |
    (F.lower(F.trim(F.col("account_status"))).isin(['unknown', 'n/a', 'none', 'null', 'na', '.'])) |
    (~F.upper(F.trim(F.col("account_status"))).isin(['ACTIVE', 'INACTIVE', 'SUSPENDED', 'DORMANT']))
).count()

print(f"Invalid account_status count: {invalid_status_count}")

Invalid account_status count: 0


signup_date

In [89]:
from pyspark.sql.types import DateType
bronze_customers.filter(
    "(signup_date IS NULL) OR (TRIM(signup_date) = '') OR (LOWER(TRIM(signup_date)) IN ('unknown', 'n/a', 'none', 'null', 'na', '.'))"
).show()

df_with_date = bronze_customers.withColumn('new_signup_date', F.col('signup_date').try_cast(DateType()) )

df_with_date.filter(
    "(new_signup_date IS NULL)"
).show()

df_with_date.agg(
    F.min('new_signup_date').alias('min_date'),
    F.max('new_signup_date').alias('max_date')
).show()



+-----------+----------+---------+-----+-------+----+-----------+----------------+--------------+--------------------+------------+
|customer_id|first_name|last_name|email|country|city|signup_date|customer_segment|account_status|_ingestion_timestamp|_source_file|
+-----------+----------+---------+-----+-------+----+-----------+----------------+--------------+--------------------+------------+
+-----------+----------+---------+-----+-------+----+-----------+----------------+--------------+--------------------+------------+

+-----------+----------+---------+--------------------+-------+------+-----------+----------------+--------------+--------------------+--------------------+---------------+
|customer_id|first_name|last_name|               email|country|  city|signup_date|customer_segment|account_status|_ingestion_timestamp|        _source_file|new_signup_date|
+-----------+----------+---------+--------------------+-------+------+-----------+----------------+--------------+-----------

## **Product Profiling**

In [90]:
bronze_products.show(5, truncate=False)

+------+-----------+----------+------------+---------+----------+-------------------+-----------+----------+--------------------------+--------------------------------------------------------------------------------+
|brand |category   |cost_price|created_date|is_active|product_id|product_name       |subcategory|unit_price|_ingestion_timestamp      |_source_file                                                                    |
+------+-----------+----------+------------+---------+----------+-------------------+-----------+----------+--------------------------+--------------------------------------------------------------------------------+
|Nova  |Beauty     |14.01     |2025-05-01  |true     |P00001    |Orion Wellness 0001|Wellness   |22.83     |2026-10-02 14:04:52.730378|file:///content/drive/MyDrive/pyspark-ecommerce-analytics-data/raw/products.json|
|Orion |Electronics|140.59    |2024-09-06  |true     |P00002    |Atlas Laptops 0002 |Laptops    |253.71    |2026-10-02 14:04:52.7303

product_id

In [91]:
bronze_products.filter((F.col('product_id').isNull()) |
 (F.trim(F.col('product_id')) == '') |
  (F.col('product_id') != F.trim(F.col('product_id'))) |
   (F.lower(F.trim(F.col('product_id'))).isin('unknown', 'n/a', 'na', 'null', 'none'))
   ).count()

0

In [92]:
bronze_products.groupBy('product_id').count().filter("count > 1").select('product_id').count()

1

product_name

In [93]:
bronze_products.filter(
    "(product_name IS NULL) OR (TRIM(product_name) = '') OR (LOWER(TRIM(product_name)) IN ('unknown', 'n/a', 'none', 'null', 'na', '.'))"
).count()

0

category

In [94]:
bronze_products.filter(
    "(category IS NULL) OR (TRIM(category) = '') OR (LOWER(TRIM(category)) IN ('unknown', 'n/a', 'none', 'null', 'na', '.'))"
).count()

1

subcategory

In [95]:
bronze_products.filter(
    "(subcategory IS NULL) OR (TRIM(subcategory) = '') OR (LOWER(TRIM(subcategory)) IN ('unknown', 'n/a', 'none', 'null', 'na', '.'))"
).count()

0

brand

In [96]:
bronze_products.filter(
    "(brand IS NULL) OR (TRIM(brand) = '') OR (LOWER(TRIM(brand)) IN ('unknown', 'n/a', 'none', 'null', 'na', '.'))"
).count()

0

unit_price

In [134]:
from pyspark.sql.types import DoubleType

double_unit_price = bronze_products.withColumn('int_unit_price',F.col('unit_price').try_cast(DoubleType()))

double_unit_price.select(
    F.count('*').alias('total_rows'),
    F.count(F.when(F.col('int_unit_price').isNull(),1)).alias('null_count'),
    F.count(F.when(F.col('int_unit_price') < 0,1)).alias('negative_count'),
    F.count(F.when(F.col('int_unit_price') == 0,1)).alias('zero_count'),
    F.min('int_unit_price').alias('min_unit_price'),
    F.max('int_unit_price').alias('max_unit_price')
).show()

+----------+----------+--------------+----------+--------------+--------------+
|total_rows|null_count|negative_count|zero_count|min_unit_price|max_unit_price|
+----------+----------+--------------+----------+--------------+--------------+
|      1000|         1|             0|         0|          7.48|        319.35|
+----------+----------+--------------+----------+--------------+--------------+



cost_price

In [135]:
from pyspark.sql.types import DoubleType

double_cost_price = bronze_products.withColumn('int_cost_price',F.col('cost_price').try_cast(DoubleType()))

double_cost_price.select(
    F.count('*').alias('total_rows'),
    F.count(F.when(F.col('int_cost_price').isNull(),1)).alias('null_count'),
    F.count(F.when(F.col('int_cost_price') < 0,1)).alias('negative_count'),
    F.count(F.when(F.col('int_cost_price') == 0,1)).alias('zero_count'),
    F.min('int_cost_price').alias('min_cost_price'),
    F.max('int_cost_price').alias('max_cost_price')
).show()

+----------+----------+--------------+----------+--------------+--------------+
|total_rows|null_count|negative_count|zero_count|min_cost_price|max_cost_price|
+----------+----------+--------------+----------+--------------+--------------+
|      1000|         1|             0|         0|          5.18|        179.94|
+----------+----------+--------------+----------+--------------+--------------+



created_date

In [99]:
from pyspark.sql.types import DateType

df_with_created_date = bronze_products.withColumn('date', F.col('created_date').try_cast(DateType()))

df_with_created_date.select(
    F.count(F.when(F.col('date').isNull(),1)).alias('null_count'),
    F.min('date').alias('min_date'),
    F.max('date').alias('max_date')
).show()

+----------+----------+----------+
|null_count|  min_date|  max_date|
+----------+----------+----------+
|         1|2020-01-02|2025-06-29|
+----------+----------+----------+



## **Order Profiling**

In [100]:
bronze_orders.show(5, truncate=False)

+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+--------------------------+-----------------------------------------------------------------------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_country|shipping_city|order_total|_ingestion_timestamp      |_source_file                                                                 |
+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+--------------------------+-----------------------------------------------------------------------------+
|O0000001|C002471    |2024-10-27|Completed   |Card          |Australia       |Melbourne    |1510.72    |2026-10-02 14:04:49.785937|file:///content/drive/MyDrive/pyspark-ecommerce-analytics-data/raw/orders.csv|
|O0000002|C009466    |2024-04-16|Completed   |UPI           |United Kingdom  |Manchester   |1716.70    |2026-10-02 14:04:49.785937|file:///content/drive/MyDrive

order_id

In [101]:
bronze_orders.groupBy('order_id').count().filter("count > 1").select('order_id').count()

1

In [102]:
bronze_orders.filter((F.col('order_id').isNull()) |
 (F.trim(F.col('order_id')) == '') |
  (F.col('order_id') != F.trim(F.col('order_id'))) |
   (F.lower(F.trim(F.col('order_id'))).isin('unknown', 'n/a', 'na', 'null', 'none'))
   ).count()

0

customer_id

In [103]:
bronze_orders.filter((F.col('customer_id').isNull()) |
 (F.trim(F.col('customer_id')) == '') |
  (F.col('customer_id') != F.trim(F.col('customer_id'))) |
   (F.lower(F.trim(F.col('customer_id'))).isin('unknown', 'n/a', 'na', 'null', 'none'))
   ).count()

0

order_date

In [104]:
from pyspark.sql.types import DateType

df_with_order_date = bronze_orders.withColumn('date', F.col('order_date').try_cast(DateType()))

df_with_order_date.select(
    F.count(F.when(F.col('date').isNull(),1)).alias('null_count'),
    F.min('date').alias('min_date'),
    F.max('date').alias('max_date')
).show()

+----------+----------+----------+
|null_count|  min_date|  max_date|
+----------+----------+----------+
|         1|2024-01-01|2025-12-31|
+----------+----------+----------+



order_status

In [105]:
bronze_orders.filter(
    (F.col('order_status').isNull()) | (F.col('order_status') != F.trim(F.col('order_status'))) | (F.lower(F.trim(F.col('order_status'))).isin('na', 'n/a', 'unknown', 'null', '', '.'))
).count()

0

shipping_country

In [106]:
bronze_orders.filter(
    (F.col('shipping_country').isNull()) | (F.col('shipping_country') != F.trim(F.col('shipping_country'))) | (F.lower(F.trim(F.col('shipping_country'))).isin('na', 'n/a', 'unknown', 'null', '', '.'))
).count()

1

shipping_city

In [107]:
bronze_orders.filter(
    (F.col('shipping_city').isNull()) | (F.col('shipping_city') != F.trim(F.col('shipping_city'))) | (F.lower(F.trim(F.col('shipping_city'))).isin('na', 'n/a', 'unknown', 'null', '', '.'))
).count()

2

payment_method

In [108]:
bronze_orders.filter(
    (F.col('payment_method').isNull()) | (F.col('payment_method') != F.trim(F.col('payment_method'))) | (F.lower(F.trim(F.col('payment_method'))).isin('na', 'n/a', 'unknown', 'null', '', '.'))
).count()

0

order_total

In [109]:
from pyspark.sql.types import DoubleType

double_order_total = bronze_orders.withColumn('double_order_total',F.col('order_total').try_cast(DoubleType()))

double_order_total.select(
    F.count('*').alias('total_rows'),
    F.count(F.when(F.col('double_order_total').isNull(),1)).alias('null_count'),
    F.count(F.when(F.col('double_order_total') < 0,1)).alias('negative_count'),
    F.count(F.when(F.col('double_order_total') < 1,1)).alias('zero_count'),
    F.min('double_order_total').alias('min_order_total'),
    F.max('double_order_total').alias('max_order_total')
).show()

+----------+----------+--------------+----------+---------------+---------------+
|total_rows|null_count|negative_count|zero_count|min_order_total|max_order_total|
+----------+----------+--------------+----------+---------------+---------------+
|     30000|         1|             1|         1|          -45.0|        2499.97|
+----------+----------+--------------+----------+---------------+---------------+



## **Transaction Profiling**

In [110]:
bronze_transactions.show(5, truncate=False)

+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+--------------------------+-----------------------------------------------------------------------------------+
|transaction_id|order_id|product_id|transaction_date|quantity|unit_price|discount_amount|tax_amount|transaction_amount|payment_status|_ingestion_timestamp      |_source_file                                                                       |
+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+--------------------------+-----------------------------------------------------------------------------------+
|T00000001     |O0028299|P00169    |2024-04-04      |2       |82.22     |16.44          |11.84     |159.84            |Captured      |2026-10-02 14:04:56.547156|file:///content/drive/MyDrive/pyspark-ecommerce-analytics-data/raw/transactions.csv|
|T00000002     |

transaction_id

In [111]:
bronze_transactions.filter((F.col('transaction_id').isNull()) |
 (F.trim(F.col('transaction_id')) == '') |
  (F.col('transaction_id') != F.trim(F.col('transaction_id'))) |
   (F.lower(F.trim(F.col('transaction_id'))).isin('unknown', 'n/a', 'na', 'null', 'none'))
   ).count()

0

order_id

In [112]:
bronze_transactions.filter((F.col('order_id').isNull()) |
 (F.trim(F.col('order_id')) == '') |
  (F.col('order_id') != F.trim(F.col('order_id'))) |
   (F.lower(F.trim(F.col('order_id'))).isin('unknown', 'n/a', 'na', 'null', 'none'))
   ).count()

0

product_id

In [113]:
bronze_transactions.filter((F.col('product_id').isNull()) |
 (F.trim(F.col('product_id')) == '') |
  (F.col('product_id') != F.trim(F.col('product_id'))) |
   (F.lower(F.trim(F.col('product_id'))).isin('unknown', 'n/a', 'na', 'null', 'none'))
   ).count()

1

transaction_date

In [152]:
from pyspark.sql.types import DateType

df_with_transaction_date = bronze_transactions.withColumn('date', F.col('transaction_date').try_cast(DateType()))

df_with_transaction_date.select(
    F.count(F.when(F.col('date').isNull(),1)).alias('null_count'),
    F.min('date').alias('min_date'),
    F.max('date').alias('max_date')
).show()

+----------+----------+----------+
|null_count|  min_date|  max_date|
+----------+----------+----------+
|         2|2024-01-01|2025-12-31|
+----------+----------+----------+



quantity

In [136]:
from pyspark.sql.types import IntegerType

int_quantity = bronze_transactions.withColumn('int_quantity', F.col('quantity').try_cast(IntegerType()))

int_quantity.select(
    F.count('*').alias('total_rows'),
    F.count(F.when(F.col('int_quantity').isNull(),1)).alias('null_count'),
    F.count(F.when(F.col('int_quantity') < 0,1)).alias('negative_count'),
    F.count(F.when(F.col('int_quantity') == 0,1)).alias('zero_count'),
    F.min('int_quantity').alias('min_quantity'),
    F.max('int_quantity').alias('max_quantity')
).show()

+----------+----------+--------------+----------+------------+------------+
|total_rows|null_count|negative_count|zero_count|min_quantity|max_quantity|
+----------+----------+--------------+----------+------------+------------+
|     50000|         1|             1|         1|          -2|           5|
+----------+----------+--------------+----------+------------+------------+



unit_price

In [116]:
from pyspark.sql.types import DoubleType

double_unit_price = bronze_transactions.withColumn('double_unit_price', F.col('unit_price').try_cast(DoubleType()))

double_unit_price.select(
    F.count('*').alias('total_rows'),
    F.count(F.when(F.col('double_unit_price').isNull(),1)).alias('null_count'),
    F.count(F.when(F.col('double_unit_price') < 0,1)).alias('negative_count'),
    F.count(F.when(F.col('double_unit_price') == 0,1)).alias('zero_count'),
    F.min('double_unit_price').alias('min_unit_price'),
    F.max('double_unit_price').alias('max_unit_price')
).show()

+----------+----------+--------------+----------+--------------+--------------+
|total_rows|null_count|negative_count|zero_count|min_unit_price|max_unit_price|
+----------+----------+--------------+----------+--------------+--------------+
|     50000|         1|             0|         0|           5.0|         300.0|
+----------+----------+--------------+----------+--------------+--------------+



discount_amount

In [117]:
from pyspark.sql.types import DoubleType

double_discount_amount = bronze_transactions.withColumn('double_discount_amount', F.col('discount_amount').try_cast(DoubleType()))

double_discount_amount.select(
    F.count('*').alias('total_rows'),
    F.count(F.when(F.col('double_discount_amount').isNull(),1)).alias('null_count'),
    F.count(F.when(F.col('double_discount_amount') < 0,1)).alias('negative_count'),
    F.min('double_discount_amount').alias('min_discount_amount'),
    F.max('double_discount_amount').alias('max_discount_amount')
).show()

+----------+----------+--------------+-------------------+-------------------+
|total_rows|null_count|negative_count|min_discount_amount|max_discount_amount|
+----------+----------+--------------+-------------------+-------------------+
|     50000|         1|             0|                0.0|             224.99|
+----------+----------+--------------+-------------------+-------------------+



tax_amount

In [133]:
from pyspark.sql.types import DoubleType

double_tax_amount = bronze_transactions.withColumn('double_tax_amount', F.col('tax_amount').try_cast(DoubleType()))

double_tax_amount.select(
    F.count('*').alias('total_rows'),
    F.count(F.when(F.col('double_tax_amount').isNull(),1)).alias('null_count'),
    F.count(F.when(F.col('double_tax_amount') < 0,1)).alias('negative_count'),
    F.min('double_tax_amount').alias('min_tax_amount'),
    F.max('double_tax_amount').alias('max_tax_amount')
).show()

+----------+----------+--------------+--------------+--------------+
|total_rows|null_count|negative_count|min_tax_amount|max_tax_amount|
+----------+----------+--------------+--------------+--------------+
|     50000|         0|             1|          -1.0|        119.94|
+----------+----------+--------------+--------------+--------------+



transaction_amount

In [119]:
from pyspark.sql.types import DoubleType

double_transaction_amount = bronze_transactions.withColumn('double_transaction_amount', F.col('transaction_amount').try_cast(DoubleType()))

double_transaction_amount.select(
    F.count('*').alias('total_rows'),
    F.count(F.when(F.col('double_transaction_amount').isNull(),1)).alias('null_count'),
    F.count(F.when(F.col('double_transaction_amount') < 0,1)).alias('negative_count'),
    F.count(F.when(F.col('double_transaction_amount') == 0,1)).alias('zero_count'),
    F.min('double_transaction_amount').alias('min_transaction_amount'),
    F.max('double_transaction_amount').alias('max_transaction_amount')
).show()

+----------+----------+--------------+----------+----------------------+----------------------+
|total_rows|null_count|negative_count|zero_count|min_transaction_amount|max_transaction_amount|
+----------+----------+--------------+----------+----------------------+----------------------+
|     50000|         0|             0|         0|                  4.63|             999999.99|
+----------+----------+--------------+----------+----------------------+----------------------+



payment_status

In [120]:
bronze_transactions.filter((F.col('payment_status').isNull()) |
 (F.trim(F.col('payment_status')) == '') |
  (F.col('payment_status') != F.trim(F.col('payment_status'))) |
   (F.lower(F.trim(F.col('payment_status'))).isin('unknown', 'n/a', 'na', 'null', 'none'))
   ).count()

0

## **Referential Integrity Profiling**

Orphan Orders

In [121]:
bronze_orders.join(
    bronze_customers,
    'customer_id',
    'left_anti'
).count()

5

Orphan Transaction

In [122]:
bronze_transactions.join(
    bronze_orders,
    'order_id',
    'left_anti'
).count()

1

Orphan Products

In [123]:
bronze_transactions.join(
    bronze_products,
    'product_id',
    'left_anti'
).count()

49

Product price vs cost

In [126]:
bronze_products.withColumn(
    'cost_price',
    F.col('cost_price').try_cast(DoubleType())
).withColumn(
    'unit_price',
    F.col('unit_price').try_cast(DoubleType())
).filter("cost_price > unit_price").count()

0

Data Relationship

In [140]:
bronze_transactions.join(
    bronze_orders,
    'order_id',
    'inner'
).filter(bronze_transactions.transaction_date > bronze_orders.order_date).show()

+--------+--------------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+--------------------+--------------------+-----------+----------+------------+--------------+----------------+-------------+-----------+--------------------+--------------------+
|order_id|transaction_id|product_id|transaction_date|quantity|unit_price|discount_amount|tax_amount|transaction_amount|payment_status|_ingestion_timestamp|        _source_file|customer_id|order_date|order_status|payment_method|shipping_country|shipping_city|order_total|_ingestion_timestamp|        _source_file|
+--------+--------------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+--------------------+--------------------+-----------+----------+------------+--------------+----------------+-------------+-----------+--------------------+--------------------+
|O0012419|     T00011500|    P00987|      2025-14-01|       2

In [183]:
df_date_customer = bronze_customers.withColumn(
    'signup_date_temp',
    F.col('signup_date').try_cast(DateType())
)

df_date_order = bronze_orders.withColumn(
    'order_date_temp',
    F.col('order_date').try_cast(DateType())
)

df_date_customer.join(
    df_date_order,
    'customer_id',
    'inner'
).filter(F.col('order_date_temp') < F.col('signup_date_temp')).select('order_date_temp', 'signup_date_temp').show(10)



+---------------+----------------+
|order_date_temp|signup_date_temp|
+---------------+----------------+
|     2024-01-01|      2025-09-18|
|     2024-04-13|      2024-10-19|
|     2025-02-26|      2025-06-06|
|     2024-04-30|      2024-12-23|
|     2024-12-14|      2025-04-19|
|     2024-08-20|      2025-10-25|
|     2025-10-21|      2025-12-11|
|     2025-04-04|      2025-07-26|
|     2024-05-31|      2025-12-18|
|     2024-10-11|      2025-01-31|
+---------------+----------------+
only showing top 10 rows


## **Referential + Grain Checks**

Customers


In [129]:
df_count = bronze_customers.count()
distinct_count = bronze_customers.select('customer_id').distinct().count()

if df_count == distinct_count:
  print('All customer_id are unique')
else:
  print(f'Duplicates found! Total Rows: {df_count} Distinct customer_ids: {distinct_count}')

Duplicates found! Total Rows: 10000 Distinct customer_ids: 9999


Orders

In [130]:
df_count = bronze_orders.count()
distinct_count = bronze_orders.select('order_id').distinct().count()

if df_count == distinct_count:
  print('All order_id are unique')
else:
  print(f'Duplicates found! Total Rows: {df_count} Distinct order_ids: {distinct_count}')

Duplicates found! Total Rows: 30000 Distinct order_ids: 29999


Products

In [131]:
df_count = bronze_products.count()
distinct_count = bronze_products.select('product_id').distinct().count()

if df_count == distinct_count:
  print('All product_id are unique')
else:
  print(f'Duplicates found! Total Rows: {df_count} Distinct product_ids: {distinct_count}')

Duplicates found! Total Rows: 1000 Distinct product_ids: 999


Transaction

In [132]:
df_count = bronze_transactions.count()
distinct_count = bronze_transactions.select('transaction_id').distinct().count()

if df_count == distinct_count:
  print('All transaction_id are unique')
else:
  print(f'Duplicates found! Total Rows: {df_count} Distinct transaction_ids: {distinct_count}')

Duplicates found! Total Rows: 50000 Distinct transaction_ids: 49999


## 11. **Data Quality Summary**

| Dataset | Rule | Invalid Count | Severity | Action |
| :--- | :--- | :--- | :--- | :--- |
| **Customers** | Missing `customer_id`? | 0 | Critical | Quarantine |
| **Customers** | Duplicate `customer_id`? | 1 | High | Deduplicate |
| **Customers** | Invalid email? | 1 | Medium | Quarantine |
| **Products** | Negative price? | 0 | High | Quarantine |
| **Orders** | Orphan customer? | 5 | Critical | Quarantine |
| **Transactions** | Zero quantity? | 1| High | Quarantine |
| **Transactions** | Orphan product? | 49 | Critical | Quarantine |

## **Silver Transformation Decisions**

Based on the profiling results:

### Standardization
- Trim identifiers and categorical fields.
- Normalize appropriate categorical values.
- Convert valid date strings to DateType.

### Validation
- Required identifiers must be non-null/non-blank.
- Numeric business fields must satisfy defined constraints.
- Status fields must belong to approved values.
- Foreign keys must reference valid parent records.

### Deduplication
- Business keys will be deduplicated using deterministic rules.

### Invalid Records
- Invalid records will not be silently dropped.
- They will be separated into quarantine datasets.

### Optional Fields
- Nullable business fields such as coupon_code will not
  automatically be treated as invalid.